In [ ]:
import kagglehub
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(f"{path}/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop("Order_ID", axis=1)
df.head()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

# since the missing values are approx less than 10% of our dataset for each col including target col its safe to drop them
df = df.dropna(subset=["Weather", "Traffic_Level", "Time_of_Day", "Courier_Experience_yrs", "Delivery_Time"])

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:

categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

# Encoding categorical columns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df.head()




In [ ]:
# Task 5: Write your code here:

from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
# Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
import seaborn as sns

print(df["Delivery_Time"].value_counts(normalize=True))
sns.countplot(x=df["Delivery_Time"])
plt.title("Target Distribution")
plt.show()

# The value counts and graph indicate target imbalance and right skew hence its preferred to use StratifiedKFold and focus on F1 score
# even if data distribution is balanced, StratifiedKFold will act like regular KFold, so always use StratifiedKFold

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1)
y = df["Delivery_Time"]

In [ ]:
# Task 2,3,4,5: Write your code here:

# Use the correct split: KFold OR StratifiedKFold
# Train a RandomForest model
# Evaluate using MAE (Mean Absolute Error) ONLY
# Print the averaged score across all folds

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = RandomForestRegressor()

mae_scores = []

for train_idx, val_idx in skf.split(X, y):


  X_fold_train, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
  y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]

  model.fit(X_fold_train, y_fold_train)
  y_fold_pred = model.predict(X_fold_val)

  mae = mean_absolute_error(y_fold_pred, y_fold_val)

  mae_scores.append(mae)


mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")







In [ ]:
# Task 1: Write your code here:

feature_cols = ["Distance_km","Weather"	,"Traffic_Level",	"Time_of_Day",	"Vehicle_Type",	"Preparation_Time_min",	"Courier_Experience_yrs"]
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

# Plot predicted delivery time histogram

df["Delivery_Time"].hist()
plt.show()


In [ ]:
# Task Bonus: Write your code here:

